In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split as train_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score

---

**🌟 Exercise 1 : Exploratory Data Analysis**

*Instructions:*
* Load the data from CSV files
* Remove target column from the training data
* Split the data intro train/test split
* Understand the data

---

In [17]:
df = pd.read_csv('/content/dataset_heart.csv')
# 3. Проверка на пропуски
print("\nMissing values:")
print(df.isnull().sum())

# 4. Типы данных
print("\nColumn data types:")
print(df.dtypes)


Missing values:
age                                     0
sex                                     0
chest pain type                         0
resting blood pressure                  0
serum cholestoral                       0
fasting blood sugar                     0
resting electrocardiographic results    0
max heart rate                          0
exercise induced angina                 0
oldpeak                                 0
ST segment                              0
major vessels                           0
thal                                    0
heart disease                           0
dtype: int64

Column data types:
age                                       int64
sex                                       int64
chest pain type                           int64
resting blood pressure                    int64
serum cholestoral                         int64
fasting blood sugar                       int64
resting electrocardiographic results      int64
max heart rate            

In [11]:
X = df.drop('heart disease', axis=1)
y = df['heart disease'].replace({1: 0, 2: 1})

#split
X_train, X_test, y_train, y_test = train_split(X, y, test_size=0.2, random_state = 42)

# scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

---

**🌟 Exercise 2 : Logistic Regression without Grid Search**

*Instructions*

* Use the dataset to build a logistic regression model without using grid search. Split the data into training and testing sets, then train a logistic regression model and evaluate its performance on the test set.


---


In [18]:
#craete model
lg = LogisticRegression()
lg.fit(X_train, y_train)

#predict in test sample
y_pred = lg.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.9074074074074074
[[31  2]
 [ 3 18]]
              precision    recall  f1-score   support

           0       0.91      0.94      0.93        33
           1       0.90      0.86      0.88        21

    accuracy                           0.91        54
   macro avg       0.91      0.90      0.90        54
weighted avg       0.91      0.91      0.91        54



---

**🌟 Exercise 3 : Logistic Regression with Grid Search**

*Instructions*

* Build a logistic regression model using the dataset, but this time, use GridSearchCV to optimize the hyperparameters such as C and penalty.

---

In [29]:
#  One-hot encoding for categorical features
categorical_cols = [
    'chest pain type', 'ST segment', 'major vessels',
    'thal', 'resting electrocardiographic results'
]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded = df_encoded.astype(int)  # Convert boolean to int


In [30]:

X = df_encoded.drop('heart disease', axis=1)
y = df_encoded['heart disease'].replace({1: 0, 2: 1})

#  Split data
X_train, X_test, y_train, y_test = train_split(
    X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set up parameter grid for GridSearchCV
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

# best hyperparameters using cross-validation
grid = GridSearchCV(LogisticRegression(), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)

Best parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-validation score: 0.8335095137420717


---

**🌟 Exercise 4 : SVM without Grid Search**

*Instructions*

* Train a Support Vector Machine (SVM) classifier on the dataset without using grid search. Choose an appropriate kernel and set the hyperparameters manually.

---

In [31]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create SVM model (for example, with RBF kernel)
svm = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)

# Fit the model
svm.fit(X_train, y_train)

# Predict
y_pred = svm.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8703703703703703

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.91      0.90        33
           1       0.85      0.81      0.83        21

    accuracy                           0.87        54
   macro avg       0.87      0.86      0.86        54
weighted avg       0.87      0.87      0.87        54


Confusion Matrix:
 [[30  3]
 [ 4 17]]


---

**🌟 Exercise 5 : SVM with Grid Search**

*Instructions*

* Implement an SVM classifier on the dataset with GridSearchCV to find the best combination of C, kernel, and gamma hyperparameters.

---

In [35]:
# define parametrs
param_grid = {'C': [0.1, 1, 10, 100, 1000],
			'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
			'kernel': ['rbf']}

# GridSearchCV for SVM
grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3)

# Fit the grid
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV 1/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.545 total time=   0.0s
[CV 2/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.558 total time=   0.0s
[CV 3/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.535 total time=   0.0s
[CV 4/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.535 total time=   0.0s
[CV 5/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.535 total time=   0.0s
[CV 1/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.636 total time=   0.0s
[CV 2/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.651 total time=   0.0s
[CV 3/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.605 total time=   0.0s
[CV 4/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.721 total time=   0.0s
[CV 5/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.581 total time=   0.0s
[CV 1/5] END .....C=0.1, gamma=0.01, kernel=rbf;, score=0.705 total time=   0.0s
[CV 2/5] END .....C=0.1, gamma=0.01, kernel=rbf

---

**🌟 Exercise 6 : XGBoost without Grid Search**
*Instructions*
* Use the dataset to train an XGBoost classifier without hyperparameter tuning. Set the hyperparameters manually and justify your choices.

---

In [38]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create  XGBoost
xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                    eval_metric='logloss',
                    random_state=42)

xgb.fit(X_train, y_train)

# Predict on test set
y_pred = xgb.predict(X_test)

# Evaluate model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8148148148148148
Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.94      0.86        33
           1       0.87      0.62      0.72        21

    accuracy                           0.81        54
   macro avg       0.83      0.78      0.79        54
weighted avg       0.82      0.81      0.81        54

Confusion Matrix:
 [[31  2]
 [ 8 13]]


---
**🌟 Exercise 7 : XGBoost with Grid Search**

*Instructions*

* Train an XGBoost classifier on the dataset using GridSearchCV to optimize hyperparameters such as learning_rate, n_estimators, max_depth, etc.

---

In [39]:
# Import libraries
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Set parameter
param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5],
}

# Create the XGBoost
xgb = XGBClassifier(eval_metric='logloss', random_state=42)

# Create GridSearchCV object
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    verbose=3,
    n_jobs=-1
)

# Fit the model
grid.fit(X_train, y_train)

# best parameters and best score
print("Best parameters:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)

# Predict on test set with the best model
y_pred = grid.best_estimator_.predict(X_test)

# Evaluate model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
Best cross-validation score: 0.8103594080338266
Accuracy: 0.7777777777777778
Confusion Matrix:
 [[30  3]
 [ 9 12]]
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.91      0.83        33
           1       0.80      0.57      0.67        21

    accuracy                           0.78        54
   macro avg       0.78      0.74      0.75        54
weighted avg       0.78      0.78      0.77        54

